# Pharmacy Desert Risk Predcition using Machine Learning



Week 3 – Data Aggregation - Data Merging

## Setup: imports and paths

Importing required libraries and configuring the DuckDB connection. This code below samples the Parquet header without loading the full file into memory to get data overview.

In [ ]:
import duckdb
import pandas as pd
from pathlib import Path

parquet_path =  'cms_partd.parquet'

# in-memory DuckDB connection for sampling and aggregations
con = duckdb.connect(database=':memory:')
print('parquet exists ->', parquet_path.exists())
# showing columns (limit 0) to inspect header/schema
cols = con.execute(f"SELECT * FROM read_parquet('{parquet_path.as_posix()}') LIMIT 0").fetchdf().columns.tolist()
print('columns (sample):', cols)

parquet exists -> True
columns (sample): ['PRSCRBR_NPI', 'Prscrbr_Last_Org_Name', 'Prscrbr_First_Name', 'Prscrbr_MI', 'Prscrbr_Crdntls', 'Prscrbr_Gndr', 'Prscrbr_Ent_Cd', 'Prscrbr_St1', 'Prscrbr_St2', 'Prscrbr_City', 'Prscrbr_State_Abrvtn', 'Prscrbr_State_FIPS', 'Prscrbr_zip5', 'Prscrbr_RUCA', 'Prscrbr_RUCA_Desc', 'Prscrbr_Cntry', 'Prscrbr_Type', 'Prscrbr_Type_src', 'Tot_Clms', 'Tot_30day_Fills', 'Tot_Drug_Cst', 'Tot_Day_Suply', 'Tot_Benes', 'GE65_Sprsn_Flag', 'GE65_Tot_Clms', 'GE65_Tot_30day_Fills', 'GE65_Tot_Drug_Cst', 'GE65_Tot_Day_Suply', 'GE65_Bene_Sprsn_Flag', 'GE65_Tot_Benes', 'Brnd_Sprsn_Flag', 'Brnd_Tot_Clms', 'Brnd_Tot_Drug_Cst', 'Gnrc_Sprsn_Flag', 'Gnrc_Tot_Clms', 'Gnrc_Tot_Drug_Cst', 'Othr_Sprsn_Flag', 'Othr_Tot_Clms', 'Othr_Tot_Drug_Cst', 'MAPD_Sprsn_Flag', 'MAPD_Tot_Clms', 'MAPD_Tot_Drug_Cst', 'PDP_Sprsn_Flag', 'PDP_Tot_Clms', 'PDP_Tot_Drug_Cst', 'LIS_Sprsn_Flag', 'LIS_Tot_Clms', 'LIS_Drug_Cst', 'NonLIS_Sprsn_Flag', 'NonLIS_Tot_Clms', 'NonLIS_Drug_Cst', 'Opioid_Tot_Clms',

## Sampling key columns

Inspecting the columns to form the unique pharmacy identifier and also to extract location/year information.

In [ ]:
# Data Overview
sample = con.execute(
    f"SELECT Prscrbr_St1, Prscrbr_St2, Prscrbr_City, Prscrbr_State_Abrvtn, Prscrbr_State_FIPS, Prscrbr_zip5, file_origin FROM read_parquet('{parquet_path.as_posix()}') LIMIT 10"
).fetchdf()

sample

,Prscrbr_St1,Prscrbr_St2,Prscrbr_City,Prscrbr_State_Abrvtn,Prscrbr_State_FIPS,Prscrbr_zip5,file_origin
0,900 Seton Dr,None,Cumberland,MD,24.0,21502.0,medicare_partd_2014.csv
1,4126 N Holland Sylvania Rd,Suite 220,Toledo,OH,39.0,43623.0,medicare_partd_2014.csv
2,4115 Dorchester Road,Concentra Medical Center,Charleston,SC,45.0,29405.0,medicare_partd_2014.csv
3,5 Pine Cone Rd,None,Dayton,NV,32.0,89403.0,medicare_partd_2014.csv
4,Tennessee Prison For Women,3881 Stewarts Lane,Nashville,TN,47.0,37243.0,medicare_partd_2014.csv
5,456 Magee Ave,None,Patton,PA,42.0,16668.0,medicare_partd_2014.csv
6,11100 Euclid Ave,None,Cleveland,OH,39.0,44106.0,medicare_partd_2014.csv
7,12605 E 16th Ave,None,Aurora,CO,8.0,80045.0,medicare_partd_2014.csv
8,1565 Saxon Blvd Ste 202,None,Deltona,FL,12.0,32725.0,medicare_partd_2014.csv
9,1021 Park Ave,Suite 203,Quakertown,PA,42.0,18951.0,medicare_partd_2014.csv


## Aggregation in DuckDB (in manageable chunks)

Aggregating provider-level rows into a manageable county/zip-year level summary inside DuckDB while keeping only the key features. Also the ZIP→county crosswalk data was loaded and used for aggregation.

In [ ]:
# building an aggregated view (zip/year granularity) and saving to a parquet file to avoid loading everything into pandas
agg_query = f"""
SELECT
  Prscrbr_State_FIPS AS state_fips,
  Prscrbr_State_Abrvtn AS state_abbrev,
  Prscrbr_zip5 AS zip5,
  Prscrbr_City AS city,
  (regexp_replace(file_origin, '.*([0-9]{{4}}).*', '\\1'))::INTEGER AS year,
  COUNT(DISTINCT Prscrbr_St1 || '|' || Prscrbr_St2) AS pharmacies_per_zip_year,
  SUM(CAST(Tot_Clms AS DOUBLE)) AS tot_claims,
  SUM(CAST(Tot_30day_Fills AS DOUBLE)) AS tot_30day_fills,
  SUM(CAST(Tot_Drug_Cst AS DOUBLE)) AS tot_drug_cost,
  SUM(CAST(Tot_Day_Suply AS DOUBLE)) AS tot_days_supply,
  SUM(CAST(Tot_Benes AS DOUBLE)) AS tot_beneficiaries,
  AVG(CAST(Bene_Avg_Age AS DOUBLE)) AS avg_bene_age,
  SUM(CAST(Bene_Feml_Cnt AS DOUBLE)) AS tot_female_benes,
  SUM(CAST(Bene_Male_Cnt AS DOUBLE)) AS tot_male_benes,
  SUM(CAST(Bene_Race_Wht_Cnt AS DOUBLE)) AS tot_white_benes,
  SUM(CAST(Bene_Race_Black_Cnt AS DOUBLE)) AS tot_black_benes,
  SUM(CAST(Bene_Race_Hspnc_Cnt AS DOUBLE)) AS tot_hispanic_benes,
  SUM(CAST(Bene_Dual_Cnt AS DOUBLE)) AS tot_dual_eligible
FROM read_parquet('{parquet_path.as_posix()}')
GROUP BY state_fips, state_abbrev, zip5, city, year
"""

out_path = ('partd_agg_by_zip_year.parquet').as_posix()
# saving aggregated parquet since DuckDB does work on disk efficiently
con.execute(f"COPY ({agg_query}) TO '{out_path}' (FORMAT PARQUET)")
print('wrote aggregated file to', out_path)
# loading a small sample back into pandas for inspection
partd_agg_sample = con.execute(f"SELECT * FROM read_parquet('{out_path}') LIMIT 10").fetchdf()
partd_agg_sample

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

wrote aggregated file to c:/Users/USER/Documents/Work/2026/Q2/Data Practicum/Sriram Pranay Sharma Devaraju/W3/partd_agg_by_zip_year.parquet


,state_fips,state_abbrev,zip5,city,year,pharmacies_per_zip_year,tot_claims,tot_30day_fills,tot_drug_cost,tot_days_supply,tot_beneficiaries,avg_bene_age,tot_female_benes,tot_male_benes,tot_white_benes,tot_black_benes,tot_hispanic_benes,tot_dual_eligible
0,38.0,ND,58102.0,Fargo,2014,17,185013.0,287793.433333,1.929888e+07,7892563.0,28544.0,69.669487,16189.0,11674.0,26441.0,117.0,80.0,7958.0
1,25.0,MA,2215.0,Boston,2014,623,626935.0,922394.433334,1.211799e+08,25800830.0,92497.0,67.706499,47173.0,34716.0,64374.0,10028.0,2504.0,31587.0
2,12.0,FL,33180.0,Miami,2014,13,38000.0,47676.533333,4.342742e+06,1335207.0,4979.0,71.332035,2951.0,1911.0,2351.0,1242.0,1116.0,1691.0
3,6.0,CA,94110.0,San Francisco,2014,231,371999.0,563655.333335,7.293663e+07,15679304.0,38517.0,64.890631,19599.0,15724.0,10765.0,4340.0,11472.0,16703.0
4,72.0,PR,928.0,San Juan,2014,9,3067.0,3213.100000,1.146245e+05,78892.0,539.0,68.512103,236.0,172.0,0.0,0.0,506.0,17.0
5,49.0,UT,84120.0,West Valley City,2014,22,125113.0,193369.933333,8.337041e+06,5474562.0,12801.0,68.490359,7737.0,4839.0,10636.0,0.0,848.0,2629.0
6,36.0,NY,12901.0,Plattsburgh,2014,64,317074.0,531918.299998,3.309792e+07,14843425.0,35936.0,66.894653,20198.0,14451.0,33835.0,414.0,146.0,12929.0
7,55.0,WI,54481.0,Stevens Point,2014,9,164919.0,284433.566666,1.513099e+07,8015885.0,20363.0,67.972378,11860.0,7932.0,19348.0,39.0,31.0,6216.0
8,38.0,ND,58401.0,Jamestown,2014,2,129317.0,190076.899998,8.337404e+06,5238194.0,11550.0,68.025109,7091.0,4239.0,11092.0,38.0,42.0,3238.0
9,1.0,AL,36619.0,Mobile,2014,3,31847.0,49114.266667,1.712503e+06,1388614.0,2313.0,68.135174,1214.0,1037.0,1775.0,444.0,0.0,762.0


## Target and pharmacy identifier

`Prscrbr_St1` + `Prscrbr_St2` are used as a pharmacy identifier and defined as the target, as the count of unique pharmacies per geographic unit (zip or county) per year.

In [ ]:
# loading county health ranks and USDA ERS tidy data
county_health = pd.read_csv('county_health_rank_final.csv', dtype={'FIPS': str})
county_health['FIPS'] = county_health['FIPS'].str.zfill(5)
usda = pd.read_csv('USDA_ERS_df_tidy.csv', dtype={'FIPS_Code': str})
usda['FIPS_Code'] = usda['FIPS_Code'].str.zfill(5)
print('county_health columns:', county_health.columns.tolist())
print('usda columns:', usda.columns.tolist())

county_health columns: ['Unnamed: 0', 'FIPS', 'State', 'County', 'Health_Outcome_Rank', 'Health_Factor_Rank', 'year']
usda columns: ['Unnamed: 0', 'FIPS_Code', 'State', 'Area_Name', 'year', 'labor_force', 'employed', 'income_pct_state', 'median_household_income', 'metro', 'rural_urban_code', 'unemployed', 'unemployment_rate', 'urban_influence_code']


## Mapping ZIP → County and merging datasets

Using the ZIP→county crosswalk data, `zip_county_crosswalk.csv` created from https://www2.census.gov/geo/docs/maps-data/data/rel/zcta_county_rel_10.txt, to map `zip5` to `county_fips` and aggregate to county-year.

In [ ]:
# loading aggregated zip-year file
agg_path = base / 'partd_agg_by_zip_year.parquet'
# using DuckDB to read parquet into pandas to avoid pyarrow dependency
partd_agg = con.execute(f"SELECT * FROM read_parquet('{agg_path.as_posix()}')").fetchdf()
# normalizing zip5 to remove any non-digits and zero-pad to 5
partd_agg['zip5'] = partd_agg['zip5'].astype(str).str.extract(r"(\d+)")[0].str.zfill(5)
print('partd_agg zip5 sample:', partd_agg['zip5'].head().tolist())
# ZIP->county mapping using crosswalk
crosswalk_path = 'zip_county_crosswalk.csv'
if crosswalk_path.exists():
    zipcw = pd.read_csv(crosswalk_path, dtype=str)
    # ensuring zipcw column name is 'zip5'
    if 'ZCTA5' in zipcw.columns:
        zipcw = zipcw.rename(columns={'ZCTA5':'zip5'})
    zipcw['zip5'] = zipcw['zip5'].astype(str).str.extract(r"(\d+)")[0].str.zfill(5)
    zipcw['county_fips'] = zipcw['county_fips'].astype(str).str.zfill(5)
    partd_agg = partd_agg.merge(zipcw[['zip5','county_fips']], on='zip5', how='left')
    # aggregating to county-year
    county_agg = partd_agg.groupby(['county_fips','year','state_abbrev'], as_index=False).agg({
        'pharmacies_per_zip_year':'sum', 'tot_claims':'sum','tot_30day_fills':'sum',
        'tot_drug_cost':'sum','tot_days_supply':'sum','tot_beneficiaries':'sum','avg_bene_age':'mean',
    }).rename(columns={'pharmacies_per_zip_year':'pharmacies_per_county_year'})
else:
    # aggregate to state-year as fallback option
    county_agg = partd_agg.groupby(['state_fips','year','state_abbrev'], as_index=False).agg({
        'pharmacies_per_zip_year':'sum', 'tot_claims':'sum','tot_30day_fills':'sum',
        'tot_drug_cost':'sum','tot_days_supply':'sum','tot_beneficiaries':'sum','avg_bene_age':'mean',
    }).rename(columns={'pharmacies_per_zip_year':'pharmacies_per_state_year'})

# ensuring year types align for merging
county_health['year'] = county_health['year'].astype(str)
usda['year'] = usda['year'].astype(str)
county_agg['year'] = county_agg['year'].astype(str)

# merging with county_health and usda on FIPS/year
if 'county_fips' in county_agg.columns:
    merged = county_agg.merge(county_health, left_on=['county_fips','year'], right_on=['FIPS','year'], how='left', indicator=True)
    print('merge with county_health indicator counts:', merged['_merge'].value_counts().to_dict())
    merged = merged.drop(columns=['_merge'])
    merged = merged.merge(usda, left_on=['county_fips','year'], right_on=['FIPS_Code','year'], how='left', indicator=True)
    print('merge with usda indicator counts (last merge):', merged['_merge'].value_counts().to_dict())
else:
    # merging by state and year as a fallback option
    merged = county_agg.merge(usda, left_on=['state_abbrev','year'], right_on=['State','year'], how='left', indicator=True)
    print('state-year merge indicator counts:', merged['_merge'].value_counts().to_dict())

# save final merged dataset as CSV to avoid parquet engine dependency
out_final = 'final_pharmacy_county_state_year.csv'
merged.to_csv(out_final, index=False)
print('final file written to', out_final)
merged.head()

partd_agg zip5 sample: ['58102', '02215', '33180', '94110', '00928']
merge with county_health indicator counts: {'both': 24591, 'left_only': 733, 'right_only': 0}
merge with usda indicator counts (last merge): {'both': 25200, 'left_only': 124, 'right_only': 0}
final file written to c:\Users\USER\Documents\Work\2026\Q2\Data Practicum\Sriram Pranay Sharma Devaraju\W3\final_pharmacy_county_state_year.csv


,county_fips,year,state_abbrev,pharmacies_per_county_year,tot_claims,tot_30day_fills,tot_drug_cost,tot_days_supply,tot_beneficiaries,avg_bene_age,...,labor_force,employed,income_pct_state,median_household_income,metro,rural_urban_code,unemployed,unemployment_rate,urban_influence_code,_merge
0,01001,2014,AL,18,210988.0,309350.166666,15129178.50,8634376.0,12912.0,65.163887,...,25639.0,24150.0,NaN,NaN,NaN,NaN,1489.0,5.8,NaN,both
1,01001,2015,AL,16,199619.0,315589.933333,15428420.39,8825272.0,13904.0,66.421291,...,25541.0,24206.0,NaN,NaN,NaN,NaN,1335.0,5.2,NaN,both
2,01001,2016,AL,16,214038.0,340770.499999,16947025.17,9561526.0,15990.0,63.815128,...,25710.0,24395.0,NaN,NaN,NaN,NaN,1315.0,5.1,NaN,both
3,01001,2018,AL,16,213177.0,369364.166666,23504102.26,10375534.0,18038.0,66.517913,...,26471.0,25515.0,NaN,NaN,NaN,NaN,956.0,3.6,NaN,both
4,01001,2019,AL,15,212171.0,381387.599999,26228387.81,10724283.0,19376.0,67.056924,...,26683.0,25914.0,NaN,NaN,NaN,NaN,769.0,2.9,NaN,both


## Inspecting merged dataset

In [ ]:
# Loading final CSV and showing .info()
import pandas as pd
from pathlib import Path

final_path = 'final_pharmacy_county_state_year.csv'

df_final = pd.read_csv(final_path, dtype=object)
print('Loaded', final_path.name, 'shape:', df_final.shape)
print('\nDataFrame.info():')
df_final.info()


Loaded final_pharmacy_county_state_year.csv shape: (25324, 30)

DataFrame.info():
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25324 entries, 0 to 25323
Data columns (total 30 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   county_fips                 25324 non-null  object
 1   year                        25324 non-null  object
 2   state_abbrev                25324 non-null  object
 3   pharmacies_per_county_year  25324 non-null  object
 4   tot_claims                  25324 non-null  object
 5   tot_30day_fills             25324 non-null  object
 6   tot_drug_cost               25324 non-null  object
 7   tot_days_supply             25324 non-null  object
 8   tot_beneficiaries           25324 non-null  object
 9   avg_bene_age                25324 non-null  object
 10  Unnamed: 0_x                24591 non-null  object
 11  FIPS                        24591 non-null  object
 12  State_x             

In [ ]:
# Computing and saveing missing totals per column
missing = df_final.isna().sum()
print('Missing entries per column (total):')
print(missing.to_string())

missing.to_csv(base / 'missing_totals_per_column.csv', header=['missing_count'])
print('\nSaved missing totals to', base / 'missing_totals_per_column.csv')


Missing entries per column (total):
county_fips                       0
year                              0
state_abbrev                      0
pharmacies_per_county_year        0
tot_claims                        0
tot_30day_fills                   0
tot_drug_cost                     0
tot_days_supply                   0
tot_beneficiaries                 0
avg_bene_age                      0
Unnamed: 0_x                    733
FIPS                            733
State_x                         733
County                          733
Health_Outcome_Rank             733
Health_Factor_Rank              733
Unnamed: 0_y                    124
FIPS_Code                       124
State_y                         124
Area_Name                       124
labor_force                     124
employed                        124
income_pct_state              25324
median_household_income       25324
metro                         25324
rural_urban_code              25324
unemployed                  

In [ ]:
# Computing and save missing entries per column per year
if 'year' in df_final.columns:
    miss_by_year = df_final.groupby('year').apply(lambda g: g.isna().sum()).T
    with pd.option_context('display.max_rows', None, 'display.max_columns', None):
        print('Missing entries per column per year:')
        print(miss_by_year.to_string())
    miss_by_year.to_csv(base / 'missing_per_column_per_year.csv')
    print('\nSaved per-year missingness to', base / 'missing_per_column_per_year.csv')
else:
    print('No "year" column found; skipping per-year missingness.')


Missing entries per column per year:
year                        2014  2015  2016  2018  2019  2020  2021
county_fips                    0     0     0     0     0     0     0
year                           0     0     0     0     0     0     0
state_abbrev                   0     0     0     0     0     0     0
pharmacies_per_county_year     0     0     0     0     0     0     0
tot_claims                     0     0     0     0     0     0     0
tot_30day_fills                0     0     0     0     0     0     0
tot_drug_cost                  0     0     0     0     0     0     0
tot_days_supply                0     0     0     0     0     0     0
tot_beneficiaries              0     0     0     0     0     0     0
avg_bene_age                   0     0     0     0     0     0     0
Unnamed: 0_x                 105   106   108   106   104   104   100
FIPS                         105   106   108   106   104   104   100
State_x                      105   106   108   106   104   104   1

C:\Users\USER\AppData\Local\Temp\ipykernel_5132\3019152174.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  miss_by_year = df_final.groupby('year').apply(lambda g: g.isna().sum()).T


## Cleaning Merged Dataset

In [ ]:
# Cleaning final dataset: removing 100% missing columns, extracting county from Area_Name, imputing numeric missingness
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_csv('final_pharmacy_county_state_year.csv', dtype=object)

print('Starting shape:', df.shape)

# Removing columns with 100% missing entries
fully_missing = df.columns[df.isna().sum() == len(df)].tolist()
print('Columns with 100% missing:', fully_missing)
df = df.drop(columns=fully_missing)
print('After removing 100% missing columns, shape:', df.shape)

# Extracting county name from Area_Name and fill missing County values
if 'Area_Name' in df.columns:
    # Area_Name format is typically 'County Name, State Abbrev'
    df['extracted_county'] = df['Area_Name'].str.split(',').str[0].str.strip()
    # Fill missing County with extracted county
    df['County'] = df['County'].fillna(df['extracted_county'])
    df = df.drop(columns=['extracted_county'])
    print('Extracted and filled County from Area_Name')

# Converting numeric columns and impute missing using Q1 by (year, state_abbrev, county_fips) grouping
# Identifying numeric columns (all numeric vars from the aggregation)
numeric_cols = ['pharmacies_per_county_year','tot_claims','tot_30day_fills','tot_drug_cost',
                'tot_days_supply','tot_beneficiaries','avg_bene_age','labor_force','employed',
                'unemployed','unemployment_rate']
numeric_cols = [c for c in numeric_cols if c in df.columns]

# Converting to numeric and track which had missing
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Imputing using Q1 by year, state_abbrev, county_fips grouping as baseline instead of using 0
for col in numeric_cols:
    if df[col].isna().sum() > 0:
        print(f'Imputing {col}: {df[col].isna().sum()} missing values')
        # Creating impute function using groupby transform to assign Q1 to each row's group
        q1_fill = df.groupby(['year','state_abbrev','county_fips'])[col].transform(lambda x: x.quantile(0.25))
        # Filling with group Q1, then with overall Q1 if group Q1 is also NaN
        overall_q1 = df[col].quantile(0.25)
        df[col] = df[col].fillna(q1_fill).fillna(overall_q1)

# Saving cleaned result
out_path = 'final_pharmacy_cleaned.csv'
df.to_csv(out_path, index=False)
print('Saved cleaned dataset to', out_path)
print('Final shape:', df.shape)
print('\nRemaining missing counts:',)
print(df.isna().sum()[df.isna().sum() > 0].to_string())

Starting shape: (25324, 30)
Columns with 100% missing: ['income_pct_state', 'median_household_income', 'metro', 'rural_urban_code', 'urban_influence_code']
After removing 100% missing columns, shape: (25324, 25)
Extracted and filled County from Area_Name
Imputing labor_force: 124 missing values
Imputing employed: 124 missing values
Imputing unemployed: 124 missing values
Imputing unemployment_rate: 124 missing values
Saved cleaned dataset to c:\Users\USER\Documents\Work\2026\Q2\Data Practicum\Sriram Pranay Sharma Devaraju\W3\final_pharmacy_cleaned.csv
Final shape: (25324, 25)

Remaining missing counts:
Unnamed: 0_x           733
FIPS                   733
State_x                733
County                 113
Health_Outcome_Rank    733
Health_Factor_Rank     733
Unnamed: 0_y           124
FIPS_Code              124
State_y                124
Area_Name              124


In [ ]:
# Cleaning health outcome rank and health factor rank variables

# Columns to clean
cols = ['Health_Outcome_Rank', 'Health_Factor_Rank']

# Replacing "NR" with NaN
df[cols] = df[cols].replace('NR', np.nan)

# Converting to numeric
df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')

# Filling missing values with Q1 (25th percentile)
# grouped by year, state, county
group_cols = ['year', 'state_abbrev', 'County']

df[cols] = (
    df.groupby(group_cols)[cols]
      .transform(lambda x: x.fillna(x.quantile(0.25)))
)

# filling remaining NaNs with overall Q1
for col in cols:
    df[col] = df[col].fillna(df[col].quantile(0.25))

# Check dtypes
print(df[cols].dtypes)

In [ ]:
# Checking remaining missing entries
df.isna().sum()

In [ ]:
df.columns

In [ ]:
# Variable selection
df = df[['State_y','County','county_fips', 'year', 'pharmacies_per_county_year', 'tot_claims', 'tot_30day_fills', 'tot_drug_cost', 'tot_days_supply',
       'tot_beneficiaries', 'avg_bene_age','Health_Outcome_Rank', 'Health_Factor_Rank', 'labor_force', 'employed', 'unemployed', 'unemployment_rate']]

In [ ]:
# Re-checking missing entries
df.isna().sum()

In [ ]:
# Removing 113 missing entries
df = df.dropna()
df.info()

In [ ]:
df.to_csv("df_final_pharmacy_cleaned.csv", index=False)

## Final Output Summary

This notebook completed the Week 2 data collection and initial cleaning process for the Pharmacy Desert Risk Prediction project.

Completed steps:
• Collected CMS Medicare Part D pharmacy utilization data.
• Processed County Health Rankings data for health outcome and health factor ranks.
• Processed USDA ERS economic data including employment, labor force, and unemployment.
• Prepared cleaned data files for later aggregation, merging, feature engineering, EDA, and modeling.
• Large raw CMS and Parquet files are not included in GitHub due to file size.

Next step:
The next notebook will aggregate and merge the cleaned datasets at the county-year level, then create PAP, TDF, and the Desert Formation Risk label.